In [18]:
from src.data.load_data import load_reviews

df_reviews_rag = load_reviews()

print(f"Total de registros: {len(df_reviews_rag)}")
print(
    "Pedidos únicos:",
    df_reviews_rag["order_id"].nunique()
)

display(df_reviews_rag.head())

Total de registros: 40950
Pedidos únicos: 40809


,review_id,order_id,review_score,review_comment_message
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,Recebi bem antes do prazo estipulado.
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,Parabéns lojas lannister adorei comprar pela I...
9,8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,aparelho eficiente. no site a marca do aparelh...
12,4b49719c8a200003f700d3d986ea1a19,9d6f15f95d01e79bd1349cc208361f09,4,"Mas um pouco ,travando...pelo valor ta Boa.\r\n"
15,3948b09f7c818e2d86c9a546758b2335,e51478e7e277a83743b6f9991dbfa3fb,5,"Vendedor confiável, produto ok e entrega antes..."


In [19]:
from src.rag.documents import create_documents

documents = create_documents(df_reviews_rag)

print("Quantidade de documentos:", len(documents))
print("\nExemplo de conteúdo:")
print(documents[0].page_content)

print("\nMetadados:")
print(documents[0].metadata)

Quantidade de documentos: 40950

Exemplo de conteúdo:
passage: Recebi bem antes do prazo estipulado.

Metadados:
{'review_id': 'e64fb393e7b32834bb789ff8bb30750e', 'order_id': '658677c97b385a9be170737859d3511b', 'review_score': 5}


In [20]:
from src.rag.embedding import load_embedding_model

embeddings = load_embedding_model()

textos = [
    "passage: Meu pedido chegou muito atrasado.",
    "passage: A entrega demorou mais do que o esperado."
]

vetores = embeddings.embed_documents(textos)

print("Quantidade de vetores:", len(vetores))
print("Dimensões do primeiro vetor:", len(vetores[0]))
print("Primeiros valores:", vetores[0][:5])


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 364.39it/s]


Quantidade de vetores: 2
Dimensões do primeiro vetor: 768
Primeiros valores: [0.0059566805139184, 0.03336201608181, -0.024319935590028763, 0.002808919409289956, 0.010610085912048817]


In [21]:
pergunta = "query: Quais clientes reclamaram de atrasos na entrega?"

vetor_pergunta = embeddings.embed_query(pergunta)

print("Dimensões do vetor da pergunta:", len(vetor_pergunta))

Dimensões do vetor da pergunta: 768


In [23]:
from pathlib import Path
from src.rag.embedding import load_embedding_model
from src.rag.documents import create_documents
from src.rag.vector_store import create_vector_store

PROJECT_ROOT = Path.cwd().parent

documents = create_documents(df_reviews_rag)
embeddings = load_embedding_model()

vector_store = create_vector_store(
    documents=documents,
    embeddings=embeddings,
    persist_directory=(
        PROJECT_ROOT / "data" / "processed" / "chroma_rebuilt"
    ),
)

print("Indexação concluída!")
print("Documentos processados:", len(documents))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 384.98it/s]


Indexação concluída!
Documentos processados: 40950


In [24]:
pergunta = "query: Quais clientes reclamaram sobre a qualidade do item?"

resultados = vector_store.similarity_search(
    pergunta,
    k=5,
)

for i, documento in enumerate(resultados, start=1):
    print(f"\nResultado {i}")
    print("Comentário:", documento.page_content)
    print("Nota:", documento.metadata["review_score"])
    print("Pedido:", documento.metadata["order_id"])


Resultado 1
Comentário: passage: O produto veio em duas oportunidades, mal embalado, com diversas avarias na pintura, algumas peças tortas. Precisam melhorar a embalagem e colocar avisos de "Frágil" ou não pisar.
Nota: 1
Pedido: ea884b5a25f86e605e1750bd936e0122

Resultado 2
Comentário: passage: O produto chegou mal embalado, caixa rasgada e molhada, possui defeitos na costura e tem um péssimo ajuste no banco do veículo (Voyage G6). Não recomendo!!
Nota: 1
Pedido: d23555007fbb6b414f8bb5ae6854782a

Resultado 3
Comentário: passage: O produto veio repleto de avarias e as peças não se encaixam devidamente, o que foi constatado por montador profissional. A qualidade também não é a indicada. Lastimável!
Nota: 1
Pedido: b78d95a93c70ace32ceafee76789a2f1

Resultado 4
Comentário: passage: Não recebi o produto e ainda não tive nenhuma resposta da minha reclamação. E o pior é que seria presente de 1 ano do avô da criança, o qual pagou mais de R$500,00 pela mesma.
Nota: 1
Pedido: 6e6bacc052a79d14a6

### Intergrando funções

In [25]:
from src.rag.chain import load_llm, ask_rag

llm = load_llm()

In [26]:
resultado = ask_rag(
    question="Quais problemas de entrega são relatados pelos clientes?",
    vector_store=vector_store,
    llm=llm,
    k=5,
)

print(resultado["answer"])

Os problemas de entrega relatados pelos clientes são:

- Produto não entregue: [Avaliação 2], [Avaliação 4]
- Problemas com a confirmação do pedido, resultando em atrasos na entrega: [Avaliação 3]
- Produto danificado durante a entrega (embalagem inadequada e vidros vazando): [Avaliação 5]


In [27]:
for fonte in resultado["sources"]:
    print("-" * 60)
    print("Review ID:", fonte["review_id"])
    print("Pedido:", fonte["order_id"])
    print("Nota:", fonte["review_score"])
    print("Comentário:", fonte["comment"])

------------------------------------------------------------
Review ID: 98938945a4aa26f21a5d2c3a6112b745
Pedido: 577b75217094b43a3e032b217d67f70f
Nota: 5
Comentário: Não tive problemas com a entrega nem com o produto, o problema foi com o pagamento que passou 2 vezes no meu cartao, fiz uma reclamação e não tive resposta. Ainda estou aguardando.
------------------------------------------------------------
Review ID: 7c8154d29d239b02ad0e8f2dcdf3261a
Pedido: cacb1996e2de4ef6334c5223259544e0
Nota: 1
Comentário: Péssimo o procedimento. Vergonha. Nada entregue até agora. Absurdo e descaso com cliente.
------------------------------------------------------------
Review ID: bf8dad1ed02a48807cbe4027625c5bc8
Pedido: 43bd50a07cc1758b29b32804f1a6d552
Nota: 1
Comentário: Já recebi diferentes e-mails: confirmação da entrega, dificuldades para confirmar pedido, pedido ainda não confirmado e mais uma vez, de confirmação da entrega. Será que chega?
------------------------------------------------------

### Tempo de processamento da pergunta

In [ ]:

from time import perf_counter

pergunta = "Quais problemas de entrega são relatados pelos clientes?"

# Etapa 1: recuperação
inicio = perf_counter()

documentos = vector_store.similarity_search(
    f"query: {pergunta}",
    k=5,
)

tempo_busca = perf_counter() - inicio
print(f"Tempo de recuperação: {tempo_busca:.2f} segundos")
print(f"Documentos recuperados: {len(documentos)}")

# Etapa 2: geração
contexto = "\n\n".join(
    f"[Avaliação {i}]\n"
    f"Comentário: {doc.page_content.removeprefix('passage: ')}\n"
    f"Nota: {doc.metadata.get('review_score', 'N/A')}"
    for i, doc in enumerate(documentos, start=1)
)

prompt = f"""
Responda em português usando apenas as avaliações fornecidas.
Se as evidências forem insuficientes, informe isso.
Seja objetivo e cite as avaliações relevantes.

Pergunta: {pergunta}

Avaliações:
{contexto}
"""

inicio = perf_counter()

resposta = llm.invoke(prompt)

tempo_llm = perf_counter() - inicio
print(f"Tempo de geração: {tempo_llm:.2f} segundos")
print("\nResposta:")
print(resposta.content)


Tempo de recuperação: 0.95 segundos
Documentos recuperados: 5
Tempo de geração: 573.43 segundos

Resposta:
Os problemas de entrega relatados pelos clientes são:

- Produto não entregue (Avaliação 2, 3, 4 e 5)
- Produto mal embalado com vidros vazando (Avaliação 5)


### Alteração do modelo de LLM para OpenAI

O tempo para geração da resposta foi superior a 9 minutos aplicando o Qwen3 4B, logo vamos testar o processamento na API da OpenAI. 

In [31]:
from dotenv import load_dotenv
from openai import OpenAI
from time import perf_counter

load_dotenv()
client = OpenAI()

inicio = perf_counter()

response = client.responses.create(
    model="gpt-4o-mini",
    input="Quem descobriu o Brasil?",
    max_output_tokens=100,
)

print(response.output_text)
print(f"Tempo: {perf_counter() - inicio:.2f} segundos")

O Brasil foi "descoberto" por Pedro Álvares Cabral em 22 de abril de 1500, quando sua expedição, que se dirigia às Índias, chegou ao território brasileiro. No entanto, é importante lembrar que antes da chegada dos europeus, o Brasil já era habitado por diversas culturas indígenas. Portanto, a expressão "descobrimento" é um tema que gera debate e reflexão sobre a história e a visão de mundo.
Tempo: 3.62 segundos


In [32]:
from src.rag.chain import load_llm, ask_rag

llm = load_llm()

resultado = ask_rag(
    question="Quais problemas de entrega são relatados pelos clientes?",
    vector_store=vector_store,
    llm=llm,
    k=5,
)

print(resultado["answer"])

Os problemas de entrega relatados pelos clientes são:

- Produto não entregue: [Avaliação 2], [Avaliação 4]
- Problemas com a confirmação do pedido, resultando em atrasos na entrega: [Avaliação 3]
- Produto danificado durante a entrega (embalagem inadequada e vidros vazando): [Avaliação 5]


In [33]:
from time import perf_counter
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

inicio = perf_counter()

response = client.responses.create(
    model="gpt-4o-mini",
    instructions="Responda em português brasileiro, em no máximo duas frases.",
    input="Quais são os benefícios de ouvir os clientes de uma loja virtual?",
    max_output_tokens=100,
)

print(f"Tempo da API: {perf_counter() - inicio:.2f} segundos")
print(response.output_text)

if response.usage:
    print("Tokens de entrada:", response.usage.input_tokens)
    print("Tokens de saída:", response.usage.output_tokens)

Tempo da API: 4.17 segundos
Ouvir os clientes de uma loja virtual ajuda a entender suas necessidades e expectativas, permitindo melhorias nos produtos e serviços. Isso também fortalece o relacionamento com o cliente e pode aumentar a fidelização e as vendas.
Tokens de entrada: 38
Tokens de saída: 43


In [36]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI()

In [37]:

from time import perf_counter

pergunta = "Quais problemas de entrega são relatados pelos clientes?"

# 1. Recuperação no Chroma
inicio = perf_counter()

documentos = vector_store.similarity_search(
    f"query: {pergunta}",
    k=5,
)

tempo_busca = perf_counter() - inicio

# 2. Preparação do contexto
inicio = perf_counter()

contexto = "\n\n".join(
    f"[Avaliação {i}]\n"
    f"Review ID: {doc.metadata.get('review_id', 'N/A')}\n"
    f"Comentário: {doc.page_content.removeprefix('passage: ')}\n"
    f"Nota: {doc.metadata.get('review_score', 'N/A')}"
    for i, doc in enumerate(documentos, start=1)
)

tempo_contexto = perf_counter() - inicio

# 3. Chamada à API
inicio = perf_counter()

client = OpenAI()

# Exemplo de chamada
response = client.responses.create(
    model="gpt-4o-mini",
    input="Responda em português: o que é experiência do cliente?",
    max_output_tokens=100,
)

print(response.output_text)
tempo_api = perf_counter() - inicio

print(f"Recuperação: {tempo_busca:.2f}s")
print(f"Preparação do contexto: {tempo_contexto:.4f}s")
print(f"API: {tempo_api:.2f}s")
print(f"Tokens de entrada: {response.usage.input_tokens}")
print(f"Tokens de saída: {response.usage.output_tokens}")
print("\nResposta:")
print(response.output_text)


Experiência do cliente (ou Customer Experience, CX) refere-se ao conjunto de interações e percepções que um cliente tem com uma empresa ao longo de sua jornada. Isso inclui desde o primeiro contato com a marca, passando pela compra, até o pós-venda e o suporte ao cliente. A experiência do cliente abrange todos os pontos de contato, como o site, lojas físicas, atendimento ao cliente, redes sociais, entre outros.

Uma boa experiência do cliente é fundamental para fidelizar consumidores
Recuperação: 0.10s
Preparação do contexto: 0.0003s
API: 3.43s
Tokens de entrada: 20
Tokens de saída: 100

Resposta:
Experiência do cliente (ou Customer Experience, CX) refere-se ao conjunto de interações e percepções que um cliente tem com uma empresa ao longo de sua jornada. Isso inclui desde o primeiro contato com a marca, passando pela compra, até o pós-venda e o suporte ao cliente. A experiência do cliente abrange todos os pontos de contato, como o site, lojas físicas, atendimento ao cliente, redes soc

In [38]:
import importlib
import src.rag.chain as chain_module

importlib.reload(chain_module)

llm = chain_module.load_llm()

resultado = chain_module.ask_rag(
    question="Quais problemas de entrega são relatados pelos clientes?",
    vector_store=vector_store,
    llm=llm,
    k=5,
)

print(resultado["answer"])

Os problemas de entrega relatados pelos clientes nas avaliações são:

1. **Não entrega do produto**: Há clientes que afirmam não ter recebido o produto, caracterizando um descaso com o atendimento. Essa reclamação é evidente nas avaliações [Avaliação 2], [Avaliação 4] e [Avaliação 5].

2. **Confusão na comunicação**: Um cliente menciona receber diferentes e-mails relacionados à entrega, gerando insegurança sobre se o pedido realmente chegaria: "Será que chega?" [Avaliação 3].

Essas evidências indicam falhas significativas no processo de entrega e na comunicação com os clientes, embora não sejam suficientes para afirmar que tais problemas sejam comuns a todos os consumidores.


In [39]:
for fonte in resultado["sources"]:
    print("-" * 50)
    print("Avaliação:", fonte["evidence_id"])
    print("Review ID:", fonte["review_id"])
    print("Nota:", fonte["review_score"])
    print("Comentário:", fonte["comment"])

print("\nMétricas:")
for metrica, valor in resultado["metrics"].items():
    print(f"{metrica}: {valor}")

--------------------------------------------------
Avaliação: 1
Review ID: 98938945a4aa26f21a5d2c3a6112b745
Nota: 5
Comentário: Não tive problemas com a entrega nem com o produto, o problema foi com o pagamento que passou 2 vezes no meu cartao, fiz uma reclamação e não tive resposta. Ainda estou aguardando.
--------------------------------------------------
Avaliação: 2
Review ID: 7c8154d29d239b02ad0e8f2dcdf3261a
Nota: 1
Comentário: Péssimo o procedimento. Vergonha. Nada entregue até agora. Absurdo e descaso com cliente.
--------------------------------------------------
Avaliação: 3
Review ID: bf8dad1ed02a48807cbe4027625c5bc8
Nota: 1
Comentário: Já recebi diferentes e-mails: confirmação da entrega, dificuldades para confirmar pedido, pedido ainda não confirmado e mais uma vez, de confirmação da entrega. Será que chega?
--------------------------------------------------
Avaliação: 4
Review ID: 4c2ab64ffc9119884d87d90bf9d630c3
Nota: 1
Comentário: Descaso com o cliente. O produto não foi

Dado que o modelo anterior demorou mais de 9 minutos e com a aplicação do modelo da OpenAI a resposta foi gerada em 3s, decidimos por manter o uso da API OpenAI e o processamento remoto.

### Teste das respostas

In [46]:

import pandas as pd
from time import perf_counter
from pathlib import Path

from src.rag.chain import ask_rag


perguntas_teste = [
    {
        "id": "Q01",
        "categoria": "Entrega",
        "pergunta": "Quais problemas de entrega aparecem nas avaliações?",
    },
    {
        "id": "Q02",
        "categoria": "Prazo",
        "pergunta": "Quais comentários mencionam atraso na entrega?",
    },
    {
        "id": "Q03",
        "categoria": "Produto",
        "pergunta": "Quais problemas de qualidade dos produtos são relatados?",
    },
    {
        "id": "Q04",
        "categoria": "Embalagem",
        "pergunta": "O que os clientes dizem sobre as embalagens?",
    },
    {
        "id": "Q05",
        "categoria": "Elogios",
        "pergunta": "Quais aspectos da compra são elogiados?",
    },
    {
        "id": "Q06",
        "categoria": "Insatisfação",
        "pergunta": "Quais motivos aparecem nas avaliações negativas?",
    },
    {
        "id": "Q07",
        "categoria": "Satisfação",
        "pergunta": "O que os clientes valorizam nas experiências positivas?",
    },
    {
        "id": "Q08",
        "categoria": "Síntese",
        "pergunta": "Quais são os principais temas presentes nos comentários recuperados?",
    },
    {
        "id": "Q09",
        "categoria": "Evidência insuficiente",
        "pergunta": (
            "O que os clientes dizem sobre uma funcionalidade específica "
            "que não aparece nas avaliações recuperadas?"
        ),
    },
    {
        "id": "Q10",
        "categoria": "Fora do escopo",
        "pergunta": "Qual foi o faturamento mensal do iFood?",
    },
]


resultados_avaliacao = []

for teste in perguntas_teste:
    print(f"Executando {teste['id']}: {teste['pergunta']}")

    inicio = perf_counter()

    resultado = ask_rag(
        question=teste["pergunta"],
        vector_store=vector_store,
        llm=llm,
        k=5,
    )

    tempo_total = perf_counter() - inicio
    metricas = resultado.get("metrics", {})

    resultados_avaliacao.append({
        "id": teste["id"],
        "categoria": teste["categoria"],
        "pergunta": teste["pergunta"],
        "resposta": resultado["answer"],
        "fontes": resultado["sources"],
        "tempo_segundos": round(tempo_total, 2),
        "tempo_busca": metricas.get("retrieval_seconds"),
        "tempo_geracao": metricas.get("generation_seconds"),
        "tokens_entrada": metricas.get("input_tokens"),
        "tokens_saida": metricas.get("output_tokens"),
    })

    print(f"Concluído em {tempo_total:.2f} segundos\n")


df_avaliacao = pd.DataFrame(resultados_avaliacao)

display(
    df_avaliacao[
        ["id", "categoria", "pergunta", "resposta", "tempo_segundos"]
    ]
)


Executando Q01: Quais problemas de entrega aparecem nas avaliações?
Concluído em 3.08 segundos

Executando Q02: Quais comentários mencionam atraso na entrega?
Concluído em 2.60 segundos

Executando Q03: Quais problemas de qualidade dos produtos são relatados?
Concluído em 3.55 segundos

Executando Q04: O que os clientes dizem sobre as embalagens?
Concluído em 3.38 segundos

Executando Q05: Quais aspectos da compra são elogiados?
Concluído em 3.39 segundos

Executando Q06: Quais motivos aparecem nas avaliações negativas?
Concluído em 2.29 segundos

Executando Q07: O que os clientes valorizam nas experiências positivas?
Concluído em 3.81 segundos

Executando Q08: Quais são os principais temas presentes nos comentários recuperados?
Concluído em 3.52 segundos

Executando Q09: O que os clientes dizem sobre uma funcionalidade específica que não aparece nas avaliações recuperadas?
Concluído em 1.94 segundos

Executando Q10: Qual foi o faturamento mensal do iFood?
Concluído em 1.49 segundos



,id,categoria,pergunta,resposta,tempo_segundos
0,Q01,Entrega,Quais problemas de entrega aparecem nas avalia...,Os problemas de entrega identificados nas aval...,3.08
1,Q02,Prazo,Quais comentários mencionam atraso na entrega?,Os comentários que mencionam atraso na entrega...,2.60
2,Q03,Produto,Quais problemas de qualidade dos produtos são ...,Os problemas de qualidade dos produtos relatad...,3.55
3,Q04,Embalagem,O que os clientes dizem sobre as embalagens?,As avaliações sobre as embalagens apresentam a...,3.38
4,Q05,Elogios,Quais aspectos da compra são elogiados?,Os aspectos da compra que são elogiados nas av...,3.39
5,Q06,Insatisfação,Quais motivos aparecem nas avaliações negativas?,As avaliações negativas apresentam os seguinte...,2.29
6,Q07,Satisfação,O que os clientes valorizam nas experiências p...,Os clientes valorizam as seguintes característ...,3.81
7,Q08,Síntese,Quais são os principais temas presentes nos co...,Os principais temas presentes nos comentários ...,3.52
8,Q09,Evidência insuficiente,O que os clientes dizem sobre uma funcionalida...,Não é possível responder à pergunta sobre uma ...,1.94
9,Q10,Fora do escopo,Qual foi o faturamento mensal do iFood?,Não é possível responder à pergunta sobre o fa...,1.49


In [47]:

for resultado in resultados_avaliacao:
    print("=" * 80)
    print(resultado["id"], "-", resultado["pergunta"])
    print("\nResposta:")
    print(resultado["resposta"])

    print("\nFontes recuperadas:")
    for fonte in resultado["fontes"]:
        print(
            f"- Review ID: {fonte['review_id']} | "
            f"Nota: {fonte['review_score']}"
        )
        print(f"  Comentário: {fonte['comment']}")


Q01 - Quais problemas de entrega aparecem nas avaliações?

Resposta:
Os problemas de entrega identificados nas avaliações incluem:

1. **Atrasos na entrega**: Um cliente mencionou que o prazo de entrega estava atrasado, o que impactou negativamente sua avaliação [Avaliação 4].

2. **Falta de informações**: Outro cliente expressou frustração devido à ausência de informações sobre o status da entrega e a falta de posicionamento do vendedor em relação ao produto, que não foi entregue dentro do prazo [Avaliação 5].

3. **Percepção negativa sobre o tempo de entrega**: Um cliente destacou que o longo prazo para a entrega de um item simples foi um fator determinante para não recomendar a loja [Avaliação 3].

Embora a Avaliação 1 mencione preocupações sobre a entrega, o cliente ficou aliviado por o produto ter chegado antes do prazo, o que não representa um problema real de entrega.

Portanto, os principais problemas relatados concentram-se em atrasos e falta de comunicação sobre o status das 